# Pre-processing Parking Violations

Dieses Notebook lädt die NYC Parking Violations Rohdaten der Fiskaljahre 2023, 2024 und 2025 aus dem HDFS, vereinheitlicht die Datenstruktur, bereinigt zentrale Felder und speichert die verarbeiteten Daten als Parquet-Dateien im HDFS.

## Ziel

- Rohdaten aus HDFS laden
- Fiscal Year aus Dateiname ergänzen (Spalte `fiscal_year` zeigt aus welchem Quell-File der Datensatz stammt)
- zentrale Spalten auswählen und Spaltennamen in `snake_case` vereinheitlichen
- Fehlende Werte mit `Unknown` ersetzen
- `Issue Date` parsen und `issue_year`, `issue_month`, `issue_weekday` ableiten
- `fy`, `fm` und `is_complete_fy` direkt aus Issue Date ableiten (NYC-Fiskaljahr: 1. Juli - 30. Juni)
- `Violation Time` bereinigen und `violation_hour`, `violation_minute` ableiten
- fehlende Werte behandeln
- Violation Code Mapping joinen (`violation_description_official`)
- Duplikate auf `summons_number` entfernen
- Ungültige Datumswerte filtern
- bereinigte Daten partitioniert nach `fy` als Parquet speichern

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, to_date, year, month, dayofweek,
    trim, upper, coalesce, regexp_extract, when
)
from pyspark.sql import functions as F
import re

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Preprocessing") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 17:23:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 17:23:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
!hdfs dfs -ls -R /parking_violations/raw

drwxr-xr-x   - cluster supergroup          0 2026-05-07 21:53 /parking_violations/raw/2023
-rw-r--r--   1 cluster supergroup 4025724719 2026-05-07 21:53 /parking_violations/raw/2023/parking_violations_2023.csv
drwxr-xr-x   - cluster supergroup          0 2026-05-07 21:08 /parking_violations/raw/2024
-rw-r--r--   1 cluster supergroup 3002584102 2026-05-07 21:08 /parking_violations/raw/2024/parking_violations_2024.csv
drwxr-xr-x   - cluster supergroup          0 2026-05-07 20:40 /parking_violations/raw/2025
-rw-r--r--   1 cluster supergroup 3064758292 2026-05-07 20:40 /parking_violations/raw/2025/parking_violations_2025.csv


In [3]:
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

# v4.0-Pfad (bleibt als Fallback erhalten, wird in diesem Notebook nicht überschrieben)
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

# Neuer v5.0-Pfad mit Partitionierung nach fy (aus Issue Date)
processed_path_v5 = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"

raw_paths, processed_path, processed_path_v5

({2023: 'hdfs:///parking_violations/raw/2023/parking_violations_2023.csv',
  2024: 'hdfs:///parking_violations/raw/2024/parking_violations_2024.csv',
  2025: 'hdfs:///parking_violations/raw/2025/parking_violations_2025.csv'},
 'hdfs:///parking_violations/processed/parking_violations_cleaned',
 'hdfs:///parking_violations/processed/parking_violations_cleaned_v5')

In [4]:
dfs = []

for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(
        path,
        header=True,
        inferSchema=False
    ).withColumn("Fiscal Year", lit(fiscal_year))
    
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

df_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

26/05/27 17:23:49 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 3:========================================================>(75 + 1) / 76]

+-----------+--------+
|Fiscal Year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16101101|
|       2025|16559243|
+-----------+--------+



In [5]:
selected_columns = [
    "Fiscal Year",
    "Summons Number",
    "Plate ID",
    "Registration State",
    "Issue Date",
    "Violation Time",
    "Violation County",
    "Violation Precinct",
    "Street Name",
    "Vehicle Make",
    "Vehicle Body Type",
    "Violation Code",
    "Violation Description"
]

existing_columns = [c for c in selected_columns if c in df_raw.columns]

df_selected = df_raw.select(existing_columns)

df_selected.show(10, truncate=False)

+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|Fiscal Year|Summons Number|Plate ID|Registration State|Issue Date|Violation Time|Violation County|Violation Precinct|Street Name       |Vehicle Make|Vehicle Body Type|Violation Code|Violation Description|
+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|2023       |1484697303    |JER1863 |NY                |06/10/2022|1037A         |NY              |10                |W 28TH ST         |TOYOT       |SDN              |67            |NULL                 |
|2023       |1484697315    |KEV4487 |NY                |06/13/2022|1045A         |NY              |10                |27TH DR           |JEEP        |SUBN             |51      

In [6]:
def normalize_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = name.strip("_")
    return name

df_clean_names = df_selected.toDF(*[normalize_column_name(c) for c in df_selected.columns])

df_clean_names.columns

['fiscal_year',
 'summons_number',
 'plate_id',
 'registration_state',
 'issue_date',
 'violation_time',
 'violation_county',
 'violation_precinct',
 'street_name',
 'vehicle_make',
 'vehicle_body_type',
 'violation_code',
 'violation_description']

In [7]:
df_clean = df_clean_names \
    .withColumn("registration_state", upper(trim(col("registration_state")))) \
    .withColumn("vehicle_make", upper(trim(col("vehicle_make")))) \
    .withColumn("vehicle_body_type", upper(trim(col("vehicle_body_type")))) \
    .withColumn("violation_county", upper(trim(col("violation_county")))) \
    .withColumn("street_name", trim(col("street_name"))) \
    .withColumn("vehicle_make", coalesce(col("vehicle_make"), lit("Unknown"))) \
    .withColumn("vehicle_body_type", coalesce(col("vehicle_body_type"), lit("Unknown"))) \
    .withColumn("violation_county", coalesce(col("violation_county"), lit("Unknown"))) \
    .withColumn("registration_state", coalesce(col("registration_state"), lit("Unknown"))) \
    .withColumn("street_name", coalesce(col("street_name"), lit("Unknown"))) \
    .withColumn("violation_precinct", col("violation_precinct").cast("integer"))

df_clean.show(10, truncate=False)


+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|fiscal_year|summons_number|plate_id|registration_state|issue_date|violation_time|violation_county|violation_precinct|street_name       |vehicle_make|vehicle_body_type|violation_code|violation_description|
+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|2023       |1484697303    |JER1863 |NY                |06/10/2022|1037A         |NY              |10                |W 28TH ST         |TOYOT       |SDN              |67            |NULL                 |
|2023       |1484697315    |KEV4487 |NY                |06/13/2022|1045A         |NY              |10                |27TH DR           |JEEP        |SUBN             |51      

In [8]:
df_clean = df_clean.withColumn(
    "issue_date_parsed",
    to_date(col("issue_date"), "MM/dd/yyyy")
).withColumn(
    "issue_year",
    year(col("issue_date_parsed"))
).withColumn(
    "issue_month",
    month(col("issue_date_parsed"))
).withColumn(
    "issue_weekday",
    dayofweek(col("issue_date_parsed"))
)

df_clean.select(
    "fiscal_year",
    "issue_date",
    "issue_date_parsed",
    "issue_year",
    "issue_month",
    "issue_weekday"
).show(20, truncate=False)

+-----------+----------+-----------------+----------+-----------+-------------+
|fiscal_year|issue_date|issue_date_parsed|issue_year|issue_month|issue_weekday|
+-----------+----------+-----------------+----------+-----------+-------------+
|2023       |06/10/2022|2022-06-10       |2022      |6          |6            |
|2023       |06/13/2022|2022-06-13       |2022      |6          |2            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/23/2022|2022-06-23       |2022      |6          |5            |
|2023       |06/23/2022|2022-06-23       |2022      |6          |5            |
|2023       |06/20/2022|2022-06-20       |2022      |6          |2            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/25/2022|2022-06-25      

### Eigene Fiskaljahr-Definition aus Issue Date

Die Spalte `fiscal_year` aus dem Dateinamen beschreibt nur die Datenquelle. Für inhaltlich korrekte Fiskaljahr-Analysen leiten wir das Fiskaljahr direkt aus dem `issue_date_parsed` ab.

**Neue Spalten:**

| Spalte | Beschreibung |
|---|---|
| `fy` | Fiskaljahr aus Issue Date (NYC-Def.: Jul–Jun) |
| `fm` | Fiscal Month (1 = Juli, 12 = Juni) |
| `is_complete_fy` | Boolean: liegt Issue Date in einem vollständig abgedeckten FY (Jul 2022 – Jun 2025)? |

Die ursprüngliche Spalte `fiscal_year` bleibt erhalten, sodass die Herkunft (welches Quell-File) weiterhin nachvollziehbar ist. Für Analysen sollte aber `fy` verwendet werden.

In [9]:
# Eigene Fiskaljahr-Logik aus Issue Date (NYC-Definition: Jul–Jun)
df_clean = df_clean.withColumn(
    "fy",
    when(col("issue_month") >= 7, col("issue_year") + 1)
    .otherwise(col("issue_year"))
).withColumn(
    "fm",
    when(col("issue_month") >= 7, col("issue_month") - 6)
    .otherwise(col("issue_month") + 6)
).withColumn(
    "is_complete_fy",
    (col("issue_date_parsed") >= "2022-07-01") &
    (col("issue_date_parsed") <= "2025-06-30")
)

# Sanity Check: Verteilung pro fy (nur vollständige FYs)
df_clean.filter(col("is_complete_fy")).groupBy("fy").agg(
    F.min("issue_date_parsed").alias("min_date"),
    F.max("issue_date_parsed").alias("max_date"),
    F.count("*").alias("n_rows")
).orderBy("fy").show(truncate=False)

# Vergleich: Wie viele Records sind in welchem fiscal_year aber in welchem fy?
print("\nKreuztabelle fiscal_year (Quelle) vs. fy (aus Issue Date):")
df_clean.groupBy("fiscal_year", "fy").count().orderBy("fiscal_year", "fy").show(20)

+----+----------+----------+--------+
|fy  |min_date  |max_date  |n_rows  |
+----+----------+----------+--------+
|2023|2022-07-01|2023-06-30|17487985|
|2024|2023-07-01|2024-06-30|20171329|
|2025|2024-07-01|2025-06-30|16251694|
+----+----------+----------+--------+


Kreuztabelle fiscal_year (Quelle) vs. fy (aus Issue Date):


[Stage 12:======================================================> (74 + 2) / 76]

+-----------+----+-----+
|fiscal_year|  fy|count|
+-----------+----+-----+
|       2023|1972|    1|
|       2023|1973|    3|
|       2023|2000|   63|
|       2023|2001|   72|
|       2023|2003|    5|
|       2023|2004|    1|
|       2023|2005|    1|
|       2023|2006|    1|
|       2023|2008|    1|
|       2023|2009|    1|
|       2023|2010|    1|
|       2023|2012|   11|
|       2023|2013|   23|
|       2023|2014|    5|
|       2023|2015|    2|
|       2023|2017|    2|
|       2023|2018|    6|
|       2023|2019|    1|
|       2023|2020|   30|
|       2023|2021|  192|
+-----------+----+-----+
only showing top 20 rows



In [10]:
from pyspark.sql.functions import regexp_extract, when

df_clean = df_clean.withColumn(
    "violation_time_clean",
    upper(trim(col("violation_time")))
).withColumn(
    "time_hour_raw",
    regexp_extract(col("violation_time_clean"), r"^(\d{1,2})\d{2}[AP]$", 1).cast("int")
).withColumn(
    "violation_minute",
    regexp_extract(col("violation_time_clean"), r"^\d{1,2}(\d{2})[AP]$", 1).cast("int")
).withColumn(
    "time_ampm",
    regexp_extract(col("violation_time_clean"), r"^[0-9]{3,4}([AP])$", 1)
)

df_clean = df_clean.withColumn(
    "violation_minute",
    when(
        (col("violation_minute") >= 0) & (col("violation_minute") <= 59),
        col("violation_minute")
    )
).withColumn(
    "violation_hour",
    when(
        (col("time_ampm") == "A") & (col("time_hour_raw") == 12),
        0
    ).when(
        (col("time_ampm") == "A") & (col("time_hour_raw").between(1, 11)),
        col("time_hour_raw")
    ).when(
        (col("time_ampm") == "P") & (col("time_hour_raw") == 12),
        12
    ).when(
        (col("time_ampm") == "P") & (col("time_hour_raw").between(1, 11)),
        col("time_hour_raw") + 12
    )
)

In [11]:
df_clean.select(
    "violation_time",
    "violation_time_clean",
    "time_hour_raw",
    "violation_minute",
    "time_ampm",
    "violation_hour"
).show(30, truncate=False)

+--------------+--------------------+-------------+----------------+---------+--------------+
|violation_time|violation_time_clean|time_hour_raw|violation_minute|time_ampm|violation_hour|
+--------------+--------------------+-------------+----------------+---------+--------------+
|1037A         |1037A               |10           |37              |A        |10            |
|1045A         |1045A               |10           |45              |A        |10            |
|1116A         |1116A               |11           |16              |A        |11            |
|1052A         |1052A               |10           |52              |A        |10            |
|1107A         |1107A               |11           |7               |A        |11            |
|1050A         |1050A               |10           |50              |A        |10            |
|1101A         |1101A               |11           |1               |A        |11            |
|1205P         |1205P               |12           |5        

In [12]:
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("violation_time").isNull().cast("int")).alias("missing_original_violation_time"),
    spark_sum(col("violation_hour").isNull().cast("int")).alias("missing_parsed_violation_hour")
).show()

[Stage 16:======================================================> (74 + 2) / 76]

+-------------------------------+-----------------------------+
|missing_original_violation_time|missing_parsed_violation_hour|
+-------------------------------+-----------------------------+
|                           1122|                       123832|
+-------------------------------+-----------------------------+



In [13]:
df_clean.filter(
    col("violation_time").isNotNull() & col("violation_hour").isNull()
).select(
    "violation_time",
    "violation_time_clean"
).distinct().show(50, truncate=False)

[Stage 19:=======================================================>(75 + 1) / 76]

+--------------+--------------------+
|violation_time|violation_time_clean|
+--------------+--------------------+
|0001A         |0001A               |
|3355P         |3355P               |
|0035A         |0035A               |
|0019A         |0019A               |
|0053A         |0053A               |
|0052A         |0052A               |
|0024P         |0024P               |
|0023A         |0023A               |
|0059A         |0059A               |
|0006A         |0006A               |
|0010A         |0010A               |
|0510          |0510                |
|0038A         |0038A               |
|0054A         |0054A               |
|0043A         |0043A               |
|0025A         |0025A               |
|0013A         |0013A               |
|0058P         |0058P               |
|0003P         |0003P               |
|1200          |1200                |
|0018A         |0018A               |
|0015P         |0015P               |
|0031A         |0031A               |
|0003A      

In [14]:
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("issue_date_parsed").isNull().cast("int")).alias("missing_issue_date_parsed")
).show()

[Stage 22:=======================================================>(75 + 1) / 76]

+-------------------------+
|missing_issue_date_parsed|
+-------------------------+
|                     2930|
+-------------------------+



In [15]:
import pandas as pd
from pyspark.sql.functions import broadcast

mapping_file = "../../data_sample/ParkingViolationCodes_January2020.xlsx"

violation_code_mapping_pd = pd.read_excel(mapping_file)

violation_code_mapping_pd = violation_code_mapping_pd.rename(columns={
    "VIOLATION CODE": "violation_code",
    "VIOLATION DESCRIPTION": "violation_description_official",
    "Manhattan  96th St. & below\n(Fine Amount $)": "fine_manhattan_96_below",
    "All Other Areas\n(Fine Amount $)": "fine_other_areas"
})

violation_code_mapping_pd["violation_code"] = violation_code_mapping_pd["violation_code"].astype(str)
violation_code_mapping_pd["violation_description_official"] = violation_code_mapping_pd["violation_description_official"].astype(str)

violation_code_mapping = spark.createDataFrame(violation_code_mapping_pd)

violation_code_mapping.show(10, truncate=False)

[Stage 25:>                                                         (0 + 1) / 1]

+--------------+------------------------------+-----------------------+----------------+
|violation_code|violation_description_official|fine_manhattan_96_below|fine_other_areas|
+--------------+------------------------------+-----------------------+----------------+
|1             |FAILURE TO DISPLAY BUS PERMIT |515                    |515             |
|2             |NO OPERATOR NAM/ADD/PH DISPLAY|515                    |515             |
|3             |UNAUTHORIZED PASSENGER PICK-UP|515                    |515             |
|4             |BUS PARKING IN LOWER MANHATTAN|115                    |115             |
|5             |BUS LANE VIOLATION            |50                     |50              |
|6             |OVERNIGHT TRACTOR TRAILER PKG |265                    |265             |
|7             |FAILURE TO STOP AT RED LIGHT  |50                     |50              |
|8             |IDLING                        |115                    |115             |
|9             |OBSTR

In [16]:
df_clean = df_clean.join(
    broadcast(violation_code_mapping),
    on="violation_code",
    how="left"
)

df_clean.select(
    "violation_code",
    "violation_description",
    "violation_description_official",
    "fine_manhattan_96_below",
    "fine_other_areas"
).show(20, truncate=False)

# Quality Check: Violation Codes ohne Match in der Mapping-Tabelle
# → violation_description_official wäre NULL für diese Codes
unmatched = df_clean.filter(col("violation_description_official").isNull()) \
    .groupBy("violation_code").count() \
    .orderBy("count", ascending=False)

print(f"Anzahl Codes ohne Mapping-Match: {unmatched.count()}")
unmatched.show(20)


+--------------+---------------------+------------------------------+-----------------------+----------------+
|violation_code|violation_description|violation_description_official|fine_manhattan_96_below|fine_other_areas|
+--------------+---------------------+------------------------------+-----------------------+----------------+
|67            |NULL                 |PEDESTRIAN RAMP               |165                    |165             |
|51            |NULL                 |SIDEWALK                      |115                    |115             |
|63            |NULL                 |NIGHTTIME STD/ PKG IN A PARK  |95                     |95              |
|63            |NULL                 |NIGHTTIME STD/ PKG IN A PARK  |95                     |95              |
|63            |NULL                 |NIGHTTIME STD/ PKG IN A PARK  |95                     |95              |
|63            |NULL                 |NIGHTTIME STD/ PKG IN A PARK  |95                     |95              |
|

Anzahl Codes ohne Mapping-Match: 3


[Stage 36:======================================================> (74 + 2) / 76]

+--------------+-----+
|violation_code|count|
+--------------+-----+
|             0| 6697|
|            95|  685|
|            94|  401|
+--------------+-----+



In [17]:
df_clean.groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(30)

[Stage 40:======================================================> (74 + 2) / 76]

+--------------+-------+
|violation_hour|  count|
+--------------+-------+
|          NULL| 123832|
|             0| 688262|
|             1| 794154|
|             2| 624456|
|             3| 513873|
|             4| 485418|
|             5| 760619|
|             6|1660549|
|             7|3023815|
|             8|4580426|
|             9|4752997|
|            10|3875622|
|            11|4725260|
|            12|4431735|
|            13|4208068|
|            14|3854605|
|            15|3255486|
|            16|2606278|
|            17|2250221|
|            18|1672049|
|            19|1234698|
|            20|1210644|
|            21|1079335|
|            22| 952637|
|            23| 858543|
+--------------+-------+



In [18]:
df_clean_filtered = df_clean \
    .filter(col("summons_number").isNotNull()) \
    .filter(col("violation_code").isNotNull() & (trim(col("violation_code")) != "")) \
    .filter(col("issue_date_parsed").isNotNull())

df_clean_filtered.groupBy("fiscal_year").count().orderBy("fiscal_year").show()


[Stage 44:=======================================================>(75 + 1) / 76]

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16099641|
|       2025|16557773|
+-----------+--------+



In [19]:
# Duplikate entfernen: gleiche Violations können in mehreren Fiskaljahr-Files vorkommen
# (z.B. Juli–August 2023 erscheint in FY2023 und FY2024)
# summons_number ist der Primary Key → dropDuplicates darauf

count_before = df_clean_filtered.count()

df_clean_filtered = df_clean_filtered.dropDuplicates(["summons_number"])

count_after = df_clean_filtered.count()
print(f"Zeilen vor Deduplizierung: {count_before:,}")
print(f"Zeilen nach Deduplizierung: {count_after:,}")
print(f"Entfernte Duplikate:        {count_before - count_after:,}")

[Stage 53:===================================================>      (8 + 1) / 9]

Zeilen vor Deduplizierung: 54,220,652
Zeilen nach Deduplizierung: 49,969,763
Entfernte Duplikate:        4,250,889


In [20]:
# Ungültige Datumswerte filtern: nur vollständige Fiskaljahre behalten
# is_complete_fy markiert Datensätze innerhalb Jul 2022 – Jun 2025
import pyspark.sql.functions as F

count_before_date = df_clean_filtered.count()

df_clean_filtered = df_clean_filtered.filter(col("is_complete_fy") == True)

count_after_date = df_clean_filtered.count()
print(f"Zeilen vor Datumsfilter:  {count_before_date:,}")
print(f"Zeilen nach Datumsfilter: {count_after_date:,}")
print(f"Entfernte Datumsfehler:   {count_before_date - count_after_date:,}")

df_clean_filtered.groupBy("fy").count().orderBy("fy").show()


Zeilen vor Datumsfilter:  49,969,763
Zeilen nach Datumsfilter: 49,967,563
Entfernte Datumsfehler:   2,200


[Stage 71:===================================================>    (11 + 1) / 12]

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21562121|
|       2024|11848421|
|       2025|16557021|
+-----------+--------+



In [21]:
# Technische Zwischenspalten entfernen — nicht im finalen Parquet benötigt
# violation_description aus Rohdaten ist inkonsistent → violation_description_official verwenden
cols_to_drop = [
    "violation_time_clean", "time_hour_raw", "time_ampm",
    "issue_date", "violation_description"
]

# Nur droppen wenn vorhanden (defensive)
cols_to_drop = [c for c in cols_to_drop if c in df_clean_filtered.columns]

df_clean_filtered = df_clean_filtered.drop(*cols_to_drop)

print("Verbleibende Spalten im finalen Dataset:")
print(df_clean_filtered.columns)


Verbleibende Spalten im finalen Dataset:
['violation_code', 'fiscal_year', 'summons_number', 'plate_id', 'registration_state', 'violation_time', 'violation_county', 'violation_precinct', 'street_name', 'vehicle_make', 'vehicle_body_type', 'issue_date_parsed', 'issue_year', 'issue_month', 'issue_weekday', 'fy', 'fm', 'is_complete_fy', 'violation_minute', 'violation_hour', 'violation_description_official', 'fine_manhattan_96_below', 'fine_other_areas']


In [22]:
# Partitionierung nach fy (aus Issue Date) für korrekte FY-Analysen
# Ziel-Pfad bewusst neu (_v5), damit v4.0-Daten erhalten bleiben
df_clean_filtered.write.mode("overwrite") \
    .partitionBy("fy") \
    .parquet(processed_path_v5)

print(f"Geschrieben nach: {processed_path_v5}")

26/05/27 17:35:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Geschrieben nach: hdfs:///parking_violations/processed/parking_violations_cleaned_v5


In [23]:
df_processed = spark.read.parquet(processed_path_v5)

df_processed.printSchema()

print("\nVerteilung nach fy (aus Issue Date):")
df_processed.groupBy("fy").count().orderBy("fy").show()

print("\nVerteilung nach fiscal_year (Quelle):")
df_processed.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

root
 |-- violation_code: string (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- summons_number: string (nullable = true)
 |-- plate_id: string (nullable = true)
 |-- registration_state: string (nullable = true)
 |-- violation_time: string (nullable = true)
 |-- violation_county: string (nullable = true)
 |-- violation_precinct: integer (nullable = true)
 |-- street_name: string (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_body_type: string (nullable = true)
 |-- issue_date_parsed: date (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- issue_month: integer (nullable = true)
 |-- issue_weekday: integer (nullable = true)
 |-- fm: integer (nullable = true)
 |-- is_complete_fy: boolean (nullable = true)
 |-- violation_minute: integer (nullable = true)
 |-- violation_hour: integer (nullable = true)
 |-- violation_description_official: string (nullable = true)
 |-- fine_manhattan_96_below: long (nullable = true)
 |-- fine_other

+----+--------+
|  fy|   count|
+----+--------+
|2022|  306279|
|2023|17246732|
|2024|16162180|
|2025|16251490|
|2026|     882|
+----+--------+


Verteilung nach fiscal_year (Quelle):


[Stage 83:=================================================>      (22 + 3) / 25]

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21562121|
|       2024|11848421|
|       2025|16557021|
+-----------+--------+



In [24]:
from pyspark.sql.functions import sum as spark_sum

important_columns = [
    "summons_number",
    "plate_id",
    "registration_state",
    "issue_date_parsed",
    "issue_month",
    "issue_weekday",
    "violation_time",
    "violation_hour",
    "violation_minute",
    "violation_code",
    "violation_description_official",
    "fine_manhattan_96_below",
    "fine_other_areas",
    "vehicle_make",
    "vehicle_body_type",
    "violation_county",
    "fiscal_year",
    "fy",
    "fm",
    "is_complete_fy"
]

null_check = df_processed.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_columns
])

null_check.show(truncate=False)


[Stage 86:==========================================>             (19 + 4) / 25]

+--------------+--------+------------------+-----------------+-----------+-------------+--------------+--------------+----------------+--------------+------------------------------+-----------------------+----------------+------------+-----------------+----------------+-----------+
|summons_number|plate_id|registration_state|issue_date_parsed|issue_month|issue_weekday|violation_time|violation_hour|violation_minute|violation_code|violation_description_official|fine_manhattan_96_below|fine_other_areas|vehicle_make|vehicle_body_type|violation_county|fiscal_year|
+--------------+--------+------------------+-----------------+-----------+-------------+--------------+--------------+----------------+--------------+------------------------------+-----------------------+----------------+------------+-----------------+----------------+-----------+
|0             |2       |0                 |0                |0          |0            |800           |114699        |978             |0             |7

In [25]:
df_processed.groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(30)

[Stage 89:===============================>                        (14 + 4) / 25]

+--------------+-------+
|violation_hour|  count|
+--------------+-------+
|          NULL| 114699|
|             0| 629283|
|             1| 729028|
|             2| 574919|
|             3| 472768|
|             4| 445412|
|             5| 698444|
|             6|1526631|
|             7|2797308|
|             8|4225453|
|             9|4391973|
|            10|3591065|
|            11|4367214|
|            12|4084772|
|            13|3880561|
|            14|3552312|
|            15|3002091|
|            16|2393389|
|            17|2059609|
|            18|1526229|
|            19|1127158|
|            20|1116317|
|            21| 997842|
|            22| 875886|
|            23| 787200|
+--------------+-------+



In [26]:
!hdfs dfs -ls /parking_violations/processed/parking_violations_cleaned_v5

Found 6 items
-rw-r--r--   2 cluster supergroup          0 2026-05-27 17:39 /parking_violations/processed/parking_violations_cleaned_v5/_SUCCESS
drwxr-xr-x   - cluster supergroup          0 2026-05-27 17:39 /parking_violations/processed/parking_violations_cleaned_v5/fy=2022
drwxr-xr-x   - cluster supergroup          0 2026-05-27 17:39 /parking_violations/processed/parking_violations_cleaned_v5/fy=2023
drwxr-xr-x   - cluster supergroup          0 2026-05-27 17:39 /parking_violations/processed/parking_violations_cleaned_v5/fy=2024
drwxr-xr-x   - cluster supergroup          0 2026-05-27 17:39 /parking_violations/processed/parking_violations_cleaned_v5/fy=2025
drwxr-xr-x   - cluster supergroup          0 2026-05-27 17:39 /parking_violations/processed/parking_violations_cleaned_v5/fy=2026


## Ergebnis

**Das Pre-processing wurde erfolgreich durchgeführt.** Die bereinigten Daten wurden als Parquet-Dateien im HDFS gespeichert:

`hdfs:///parking_violations/processed/parking_violations_cleaned_v5`

Die Daten sind nach `fy` (aus Issue Date abgeleitet) partitioniert. Dadurch können spätere Analysen pro echtem Fiskaljahr effizienter ausgeführt werden.

Im Pre-processing wurden folgende Schritte durchgeführt:

- Zentrale Spalten ausgewählt und Spaltennamen in `snake_case` vereinheitlicht
- Textfelder wie `vehicle_make`, `vehicle_body_type`, `violation_county`, `registration_state` und `street_name` bereinigt und fehlende Werte mit `Unknown` ersetzt
- `violation_precinct` zu Integer gecastet
- `issue_date` in ein Datumsfeld umgewandelt; `issue_year`, `issue_month`, `issue_weekday` abgeleitet
- `fy`, `fm` und `is_complete_fy` direkt aus `issue_date_parsed` abgeleitet (NYC-Fiskaljahr: 1. Juli – 30. Juni)
- `violation_time` bereinigt; `violation_hour` und `violation_minute` abgeleitet — ungültige Zeitwerte bewusst als `NULL` belassen
- Violation Code Mapping gejoint (`violation_description_official`, Bussbeträge) — `violation_description` aus Rohdaten entfernt da inkonsistent
- Zeilen mit fehlender `summons_number`, fehlendem `violation_code` oder nicht parsebarem `issue_date` entfernt
- Duplikate auf `summons_number` entfernt (gleiche Violations erscheinen in mehreren Fiskaljahr-Files)
- Datensätze ausserhalb der vollständigen Fiskaljahre FY2023–FY2025 entfernt (`is_complete_fy`)
- Technische Zwischenspalten und `violation_description` entfernt
- Partitionierung im Parquet nach `fy`

Die finalen Checks zeigen:

- Wichtige Analysefelder enthalten keine fehlenden Werte
- `violation_hour` enthält nur gültige Werte von `0` bis `23` oder `NULL`
- HDFS-Output enthält `_SUCCESS` sowie Partitionen `fy=2023`, `fy=2024`, `fy=2025`

Für Tageszeit-Analysen nur Datensätze mit `violation_hour IS NOT NULL` verwenden.
Für Analysen nach Fiskaljahr `fy` und `fm` verwenden — nicht `fiscal_year` oder `issue_month`.


In [27]:
spark.stop()